# Sıfırdan modeller, 5 seed: ResNet-34, CRNN, küçük CNN
Accelerator **GPU T4 x2**, Internet **On**, Input: **musicrec-emb-mel** (log-mel dataset'i).

Her model ayrı bir commit'te koşar (tek commit 12 saatle sınırlı). Hücre 4'teki `RUN` satırını değiştirip **Save & Run All (Commit)**:

| commit | `RUN` | tahmini süre (5 seed) |
|---|---|---|
| 1 | `"resnet34"` | ~9 sa |
| 2 | `"crnn"` | ~4-7 sa (önce draft'ta duman testi, hücre 5) |
| 3 | `"small"` | ~2 sa |

Ortak rapor: son commit'ten önce önceki commit'lerin çıktılarını **Add Input > Your Work > Notebook Output** ile ekle. Görselleştirme hücresi `/kaggle/working` ve `/kaggle/input` altındaki bütün `summary.json` dosyalarını bulup tek raporda birleştirir.

In [ ]:
REPO_URL = "https://github.com/muzik-oneri/musicrec.git"
!rm -rf /kaggle/temp/musicrec
!git clone -q $REPO_URL /kaggle/temp/musicrec
!pip install -q -r /kaggle/temp/musicrec/requirements.txt
!cd /kaggle/temp/musicrec && git log -1 --format="KOD SÜRÜMÜ: %h %s" && ls src/crnn.py src/train_cnn.py

In [ ]:
import glob, os
# log-mel dataset'inin klasörü: içinde vocab.json ve mel/ olan klasör (Kaggle yolu sürüme göre değişiyor)
EMB = [os.path.dirname(p) for p in glob.glob("/kaggle/input/**/vocab.json", recursive=True)
       if os.path.isdir(os.path.join(os.path.dirname(p), "mel"))][0]
print("EMB =", EMB)
!ls $EMB

In [ ]:
# ---- BU COMMIT'TE NE KOŞACAK ----
RUN = "resnet34"          # "resnet34" | "crnn" | "small"
SEEDS = "1 2 3 4 5"

# Üç model için aynı eğitim tarifi: adil karşılaştırma (yalnızca mimari değişir)
COMMON = ("--epochs 80 --patience 15 --batch-size 64 --lr 1e-3 --weight-decay 5e-2 --crop-sec 15 "
          "--mixup 0.3 --ema 0.999 --gain-db 6 --clip 5 --pos-weight")
EXTRA = {"resnet34": "--drop-path 0.1", "crnn": "--rnn-hidden 128 --rnn-layers 2", "small": ""}
OUT = {"resnet34": "runs_resnet", "crnn": "runs_crnn", "small": "runs_cnn"}
TRAIN = "/kaggle/temp/musicrec/src/train_cnn.py"
print(RUN, SEEDS)

### Duman testi (yalnızca draft'ta, commit'ten önce silebilirsin)
500 şarkı, 2 epoch. Epoch süresine bak: tam eğitimde epoch süresi ≈ bu süre x 3-4 (eğitim adımı 20 kat, değerlendirme aynı).

In [ ]:
!python $TRAIN --emb-dir $EMB --out-dir /kaggle/working/smoke --arch $RUN {COMMON} {EXTRA[RUN]} --epochs 2 --max-train 500 --seeds 1
!rm -rf /kaggle/working/smoke

### Tam eğitim (5 seed)

In [ ]:
!python $TRAIN --emb-dir $EMB --out-dir /kaggle/working/{OUT[RUN]} --arch $RUN {COMMON} {EXTRA[RUN]} --seeds $SEEDS

### Görselleştirme: bulunan bütün koşular tek raporda
`/kaggle/working/viz/report.html` dosyasını Output sekmesinden indir.

In [ ]:
!pip install -q matplotlib
import glob, os
runs, seen = [], set()
for sj in (sorted(glob.glob("/kaggle/working/**/summary.json", recursive=True)) +
           sorted(glob.glob("/kaggle/input/**/summary.json", recursive=True))):
    d = os.path.dirname(sj)
    label = os.path.basename(d).replace("runs_", "")
    if label in seen:          # aynı koşu hem working'de hem input'ta varsa working'deki (bu commit'in) kullanılır
        continue
    seen.add(label)
    runs.append(f"{d}:{label}")
print("\n".join(runs))
RUNS = " ".join(runs)
!python /kaggle/temp/musicrec/src/visualize.py --splits-dir $EMB/splits --mel-dir $EMB/mel --runs $RUNS --out /kaggle/working/viz
!cat /kaggle/working/viz/tables/model_comparison.csv